# VeriPix - Phase 1 Results Notebook (FatFormer baseline, 8 generators)
Runs the **official FatFormer released checkpoint** on your two Kaggle datasets and produces every table and figure for the Phase 1 slides.

**Kaggle settings:** Accelerator = GPU (T4 / P100), Internet = ON.
**Attach both datasets** with *Add Input*: `veripixtestset1` (biggan, cyclegan, deepfake, gaugan) and `veripixtestset2` (progan, stargan, stylegan, stylegan2).

**Order:** Section 1 first checks the number of images for every generator. Run Sections 2-9 only after the counts look right.

**Honesty rules for the slides:** label every result "FatFormer released checkpoint, evaluated by us". Seen generator = ProGAN (the training generator); the other seven are unseen GANs. There are no diffusion-model results yet. Do not present the paper's numbers as yours.

## 1. FIRST: check the dataset (images per generator)

In [ ]:
import os, sys, time, json, random
from pathlib import Path
import numpy as np, pandas as pd

INPUT_ROOT = "/kaggle/input"
GAN_NAMES = ["progan", "stylegan", "stylegan2", "biggan", "cyclegan", "stargan", "gaugan", "deepfake"]
DM_NAMES  = ["dalle","glide_50_27","glide_100_10","glide_100_27","guided","ldm_100","ldm_200","ldm_200_cfg","PNDM"]
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def group_of(g):
    if g == "progan": return "seen"
    return "unseen GAN" if g in GAN_NAMES else "unseen diffusion"

def find_test_roots(base):
    roots = []
    for dp, dn, fn in os.walk(base):
        if dp[len(base):].count(os.sep) > 6: dn[:] = []; continue
        if os.path.basename(dp) == "test" and any(g in dn for g in GAN_NAMES + DM_NAMES):
            roots.append(dp); dn[:] = []             # do not walk into the image folders
    return sorted(roots)

ROOTS = find_test_roots(INPUT_ROOT)
print("test folders found:")
for r in ROOTS: print("  ", r)
assert ROOTS, "No 'test' folder with generator folders found. Attach both datasets with Add Input."

rows = []
for r in ROOTS:
    ds = Path(r).parent.name
    for g in sorted(os.listdir(r)):
        gd = Path(r) / g
        if not gd.is_dir(): continue
        nr = nf = 0
        for dp, dn, fn in os.walk(gd, followlinks=True):
            n = sum(1 for f in fn if os.path.splitext(f)[1].lower() in IMG_EXT)
            parts = Path(dp).relative_to(gd).parts
            if "0_real" in parts: nr += n
            elif "1_fake" in parts: nf += n
        rows.append(dict(dataset=ds, generator=g, group=group_of(g), n_real=nr, n_fake=nf, total=nr + nf, complete=(nr > 0 and nf > 0), root=r))
COUNTS = pd.DataFrame(rows)
display(COUNTS.drop(columns="root"))

ok = COUNTS[COUNTS.complete]
print("Complete generators:", sorted(ok.generator))
print("Expected GANs missing:", [g for g in GAN_NAMES if g not in set(ok.generator)])
print("Incomplete / ignored:", sorted(COUNTS[~COUNTS.complete].generator))
dups = COUNTS.generator[COUNTS.generator.duplicated()].tolist()
if dups: print("WARNING: generator in both datasets:", dups)
print(f"Total images: {int(ok.total.sum()):,} | at ~72 ms/image the full run takes about {ok.total.sum()*0.072/60:.0f} minutes")

## 2. Build the evaluation folder (links, no copying)
`EVAL_CAP` limits the images per generator and label so the run finishes in reasonable time. `None` = use everything (slower).

In [ ]:
EVAL_CAP = 2000          # None = all images; 2000 = at most 2000 real + 2000 fake per generator
SEED = 42

EVAL_BASE = Path("/kaggle/working/eval"); EVAL = EVAL_BASE / "test"
import shutil
shutil.rmtree(EVAL_BASE, ignore_errors=True); EVAL.mkdir(parents=True)
rng = random.Random(SEED)
GENS = []
for _, row in ok.drop_duplicates("generator").iterrows():
    g, src = row.generator, Path(row.root) / row.generator
    if EVAL_CAP is None:
        os.symlink(src, EVAL / g)
    else:
        items = {"0_real": [], "1_fake": []}
        for dp, dn, fn in os.walk(src, followlinks=True):
            rel = Path(dp).relative_to(src)
            lab = "0_real" if "0_real" in rel.parts else ("1_fake" if "1_fake" in rel.parts else None)
            if lab: items[lab] += [(Path(dp) / f, rel / f) for f in fn if os.path.splitext(f)[1].lower() in IMG_EXT]
        for lab, lst in items.items():
            lst = sorted(lst)
            if len(lst) > EVAL_CAP: lst = sorted(rng.sample(lst, EVAL_CAP))
            for s, rel in lst:
                d = EVAL / g / rel; d.parent.mkdir(parents=True, exist_ok=True)
                if not d.exists(): os.symlink(s, d)
    GENS.append(g)
DATA_ROOT = str(EVAL_BASE)
print("Generators used:", GENS)
print("DATA_ROOT =", DATA_ROOT)

## 3. Environment (screenshot for the feasibility slide)

In [ ]:
import torch
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory/1e9, 1), "GB")
else:
    print("WARNING: no GPU. Settings > Accelerator > GPU.")

## 4. Get the official code and install the extra packages

In [ ]:
!git clone https://github.com/Michel-liu/FatFormer.git
%cd /kaggle/working/FatFormer
!pip install -q pytorch_wavelets PyWavelets gdown ftfy regex

In [ ]:
p = "models/clip/clip.py"
t = open(p).read()
t = t.replace("from pkg_resources import packaging", "import packaging, packaging.version")
open(p, "w").write(t)
print("patched", p)

## 5. Download CLIP ViT-L/14 weights and the FatFormer checkpoint

In [ ]:
!mkdir -p pretrained ckpt
!wget -q -O pretrained/ViT-L-14.pt https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt
!gdown "https://drive.google.com/uc?id=1Q_Kgq4ygDf8XEHgAf-SgDN6Ru_IOTLkj" -O ckpt/fatformer.pth
!ls -lh pretrained ckpt

In [ ]:
CKPT_PATH = "ckpt/fatformer.pth"      # if the Drive download failed, upload the checkpoint as a Kaggle Dataset and put its path here
assert os.path.getsize(CKPT_PATH) > 1e6, "Checkpoint missing or too small"
assert os.path.getsize("pretrained/ViT-L-14.pt") > 5e8, "CLIP weights missing"
print("files OK")

## 6. Build the model and load the checkpoint

In [ ]:
%cd /kaggle/working/FatFormer
sys.path.insert(0, os.getcwd())
from main import get_args_parser
from models import build_model

args = get_args_parser().parse_args(["--test_selected_subsets", "progan",
                                     "--num_vit_adapter", "3", "--num_context_embedding", "8"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(args)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(model.load_state_dict(ck["model"]))
model = model.to(device).eval()
n_all = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_all/1e6:.1f} M")

## 7. Run inference on every generator

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from utils.dataset import Dataset_Creator

creator = Dataset_Creator(dataset_path=DATA_ROOT, batch_size=32, num_workers=2,
                          img_resolution=256, crop_resolution=224)
datasets, names = creator.build_dataset("test", selected_subsets=GENS)

def paths_of(ds):
    if hasattr(ds, "datasets"):
        return [p for d in ds.datasets for p, _ in d.samples]
    return [p for p, _ in ds.samples]

@torch.no_grad()
def run(ds):
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    ys, ps, t_total = [], [], 0.0
    for x, y in tqdm(dl, leave=False):
        x = x.to(device)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        out = model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t_total += time.time() - t0
        ps += out.softmax(dim=1)[:, 1].cpu().tolist()
        ys += y.tolist()
    return np.array(ys), np.array(ps), 1000 * t_total / max(len(ys), 1)

RESULTS = {}
for name, ds in zip(names, datasets):
    y, p, ms = run(ds)
    RESULTS[name] = dict(y=y, p=p, ms=ms, paths=paths_of(ds))
    print(f"{name:12s} images={len(y):5d}  {ms:.1f} ms/image")

## 8. Metrics tables (Results slides)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)
RES = "/kaggle/working/results"; os.makedirs(RES, exist_ok=True)

def metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(set(y.tolist())) > 1
    return dict(n_real=int((y == 0).sum()), n_fake=int((y == 1).sum()),
        acc=100*accuracy_score(y, pred),
        real_acc=100*(pred[y == 0] == 0).mean() if (y == 0).any() else np.nan,
        fake_acc=100*(pred[y == 1] == 1).mean() if (y == 1).any() else np.nan,
        precision=100*precision_score(y, pred, zero_division=0),
        recall=100*recall_score(y, pred, zero_division=0),
        f1=100*f1_score(y, pred, zero_division=0),
        ap=100*average_precision_score(y, p) if both else np.nan,
        auc=100*roc_auc_score(y, p) if both else np.nan,
        TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))

order = {g: i for i, g in enumerate(GAN_NAMES + DM_NAMES)}
rows = []
for g, r in RESULTS.items():
    m = metrics(r["y"], r["p"]); m.update(generator=g, group=group_of(g), ms_per_image=r["ms"]); rows.append(m)
df = pd.DataFrame(rows)
df = df[["generator","group","n_real","n_fake","acc","real_acc","fake_acc","precision","recall","f1","ap","auc","TN","FP","FN","TP","ms_per_image"]]
df = df.sort_values("generator", key=lambda s: s.map(order)).reset_index(drop=True)
df.round(1).to_csv(f"{RES}/results_table.csv", index=False)
display(df.round(1))

grp = df.groupby("group")[["acc","precision","recall","f1","ap","auc"]].mean().round(1)
grp["n_generators"] = df.groupby("group").size()
display(grp); grp.to_csv(f"{RES}/group_summary.csv")

dtab = df[["generator","group","n_real","n_fake"]].copy(); dtab["total"] = dtab.n_real + dtab.n_fake
display(dtab); dtab.to_csv(f"{RES}/dataset_counts_used.csv", index=False)
COUNTS.drop(columns="root").to_csv(f"{RES}/dataset_counts_all.csv", index=False)
print("Total images evaluated:", int(dtab.total.sum()))

## 9. Comparison with the paper (clearly labelled)
Paper numbers: official README, 4-class ProGAN training, **full** test sets. Check which checkpoint you downloaded before quoting them.

In [ ]:
PAPER = {"progan":(99.9,100.0),"stylegan":(97.2,99.8),"stylegan2":(98.8,100.0),"biggan":(99.5,100.0),
 "cyclegan":(99.3,100.0),"stargan":(99.8,100.0),"gaugan":(99.4,100.0),"deepfake":(93.2,98.0)}
cmp = df[["generator","acc","ap"]].copy()
cmp["paper_acc"] = cmp.generator.map(lambda g: PAPER.get(g, (np.nan, np.nan))[0])
cmp["paper_ap"]  = cmp.generator.map(lambda g: PAPER.get(g, (np.nan, np.nan))[1])
cmp["acc_diff"] = cmp.acc - cmp.paper_acc
cmp = cmp.rename(columns={"acc":"our_acc","ap":"our_ap"})
display(cmp.round(1)); cmp.round(1).to_csv(f"{RES}/comparison_with_paper.csv", index=False)

## 10. Figures for the slides (PNG files in /kaggle/working/results)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
COL = {"seen":"#0E8C8C", "unseen GAN":"#12284C", "unseen diffusion":"#B45309"}
plt.rcParams.update({"font.family":"serif", "font.size":11})

def bar(col, title, fn, lo=0):
    plt.figure(figsize=(9, 4))
    plt.bar(df.generator, df[col], color=[COL[g] for g in df.group])
    plt.ylim(lo, 101); plt.xticks(rotation=30, ha="right"); plt.ylabel(col + " (%)"); plt.title(title)
    for g, c in COL.items():
        if g in set(df.group): plt.bar(0, 0, color=c, label=g)
    plt.legend(loc="lower left"); plt.tight_layout(); plt.savefig(f"{RES}/{fn}", dpi=200); plt.show()
lo = max(0, np.floor(df.acc.min() / 10) * 10 - 10)
bar("acc", "Accuracy per generator", "fig_accuracy.png", lo)
bar("ap", "Average precision per generator", "fig_ap.png", lo)

# real vs fake accuracy per generator
x = np.arange(len(df)); w = 0.38
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, df.real_acc, w, label="real images", color="#2563EB")
plt.bar(x + w/2, df.fake_acc, w, label="fake images", color="#F97316")
plt.xticks(x, df.generator, rotation=30, ha="right"); plt.ylim(lo, 101); plt.ylabel("accuracy (%)")
plt.title("Real vs fake accuracy per generator"); plt.legend(loc="lower left"); plt.tight_layout()
plt.savefig(f"{RES}/fig_real_vs_fake_acc.png", dpi=200); plt.show()

In [ ]:
Y = np.concatenate([r["y"] for r in RESULTS.values()]); Pp = np.concatenate([r["p"] for r in RESULTS.values()])
def cm_plot(y, p, title, fn):
    cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
    plt.figure(figsize=(4, 3.6)); plt.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=13)
    plt.xticks([0, 1], ["Real", "Fake"]); plt.yticks([0, 1], ["Real", "Fake"])
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(title); plt.tight_layout()
    plt.savefig(f"{RES}/{fn}", dpi=200); plt.show()
cm_plot(Y, Pp, "Confusion matrix (all generators)", "fig_confusion_all.png")
unseen = [g for g in RESULTS if group_of(g) != "seen"]
if unseen:
    cm_plot(np.concatenate([RESULTS[g]["y"] for g in unseen]), np.concatenate([RESULTS[g]["p"] for g in unseen]),
            "Confusion matrix (unseen generators)", "fig_confusion_unseen.png")

plt.figure(figsize=(6, 5))
for g in df.generator:
    r = RESULTS[g]
    if len(set(r["y"].tolist())) > 1:
        fpr, tpr, _ = roc_curve(r["y"], r["p"]); plt.plot(fpr, tpr, label=g)
plt.plot([0, 1], [0, 1], "k--", lw=1); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves"); plt.legend(fontsize=8, ncol=2); plt.tight_layout()
plt.savefig(f"{RES}/fig_roc.png", dpi=200); plt.show()

In [ ]:
show = (["progan"] if "progan" in RESULTS else []) + [g for g in df.sort_values("acc").generator if g != "progan"][:3]
fig, ax = plt.subplots(1, len(show), figsize=(3.3 * len(show), 3), squeeze=False)
for a, g in zip(ax[0], show):
    r = RESULTS[g]
    a.hist(r["p"][r["y"] == 0], bins=20, range=(0, 1), alpha=0.7, label="real", color="#2563EB")
    a.hist(r["p"][r["y"] == 1], bins=20, range=(0, 1), alpha=0.7, label="fake", color="#F97316")
    a.set_title(g); a.set_xlabel("P(fake)")
ax[0][0].legend(); plt.suptitle("Score distributions: seen generator and the 3 weakest unseen generators", y=1.03)
plt.tight_layout(); plt.savefig(f"{RES}/fig_score_distribution.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
from PIL import Image
ALLP = [(g, pth, int(y), float(p)) for g, r in RESULTS.items() for pth, y, p in zip(r["paths"], r["y"], r["p"])]
fp = sorted([t for t in ALLP if t[2] == 0 and t[3] >= 0.5], key=lambda t: -t[3])[:4]    # real called fake
fn = sorted([t for t in ALLP if t[2] == 1 and t[3] < 0.5],  key=lambda t: t[3])[:4]      # fake called real
ok_r = [t for t in ALLP if t[2] == 0 and t[3] < 0.5][:2]; ok_f = [t for t in ALLP if t[2] == 1 and t[3] >= 0.5][:2]
sel = ok_r + ok_f + fp + fn
fig, ax = plt.subplots(2, 4, figsize=(12, 6.6))
for a in ax.ravel(): a.axis("off")
for a, (g, pth, y, p) in zip(ax.ravel(), sel[:8]):
    a.imshow(Image.open(pth).convert("RGB"))
    true = "fake" if y else "real"; pred = "fake" if p >= 0.5 else "real"
    a.set_title(f"{g}\ntrue {true} | pred {pred} ({p:.2f})", fontsize=9, color="green" if true == pred else "red")
plt.suptitle("Sample predictions: correct (green) and the most confident errors (red)"); plt.tight_layout()
plt.savefig(f"{RES}/fig_samples_errors.png", dpi=200); plt.show()

## 11. Save predictions, run info and zip everything

In [ ]:
rows = [dict(generator=g, path=pth, label=int(y), p_fake=float(p), pred=int(p >= 0.5))
        for g, r in RESULTS.items() for pth, y, p in zip(r["paths"], r["y"], r["p"])]
pd.DataFrame(rows).to_csv(f"{RES}/predictions.csv", index=False)
info = dict(torch=torch.__version__, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            params_million=round(n_all / 1e6, 1), checkpoint=CKPT_PATH, threshold=0.5, eval_cap_per_class=EVAL_CAP,
            generators=list(RESULTS))
json.dump(info, open(f"{RES}/run_info.json", "w"), indent=2)
!cd /kaggle/working && zip -qr veripix_phase1_results.zip results && ls -lh veripix_phase1_results.zip

## 12. Summary lines for the slides

In [ ]:
for name in ["seen", "unseen GAN", "unseen diffusion"]:
    sub = df[df.group == name]
    if len(sub):
        print(f"{name:17s}: mean accuracy {sub.acc.mean():.1f}%, mean AP {sub.ap.mean():.1f}%  ({len(sub)} generator(s))")
print(f"Speed: {df.ms_per_image.mean():.1f} ms per image on {info['gpu']}")
w = df.sort_values("acc").iloc[0]
print(f"Weakest generator: {w.generator} ({w.acc:.1f}% accuracy, real {w.real_acc:.1f}%, fake {w.fake_acc:.1f}%)")
print(f"Total images evaluated: {int(df.n_real.sum() + df.n_fake.sum()):,}")

## Which file goes on which slide
| File | Slide |
|---|---|
| Section 1 table, `dataset_counts_all.csv` | Dataset slide (counts per generator) |
| Section 3 screenshot, `run_info.json` | Experimental setup |
| `results_table.csv`, `group_summary.csv` | Baseline results (all generators) |
| `fig_accuracy.png`, `fig_real_vs_fake_acc.png` | Results by generator, seen vs unseen |
| `fig_confusion_all.png`, `fig_roc.png` | Evaluation |
| `fig_score_distribution.png` | Generalization analysis |
| `fig_samples_errors.png` | Sample predictions and errors |
| `comparison_with_paper.csv` | Comparison (paper vs ours) |

Caption every result **"FatFormer released checkpoint, evaluated by us"**, and state EVAL_CAP and that no diffusion models were tested.